In [ ]:
# Import libraries here
import gzip
import json
import pickle 

import matplotlib.pyplot as plt
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
from IPython.display import VimeoVideo
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from imblearn.over_sampling import RandomOverSampler


In [ ]:
# Load data file
with gzip.open("data/taiwan-bankruptcy-data.json.gz", "r") as f:
    taiwan_data = json.load(f)
print(type(taiwan_data))


In [ ]:
taiwan_data_keys = taiwan_data.keys()
print(taiwan_data_keys)


In [ ]:
n_companies = len(taiwan_data['observations'])
print(n_companies)


In [ ]:
n_features = len(taiwan_data['observations'][0])
print(n_features)


In [ ]:
# Create wrangle function
def wrangle(filename):
    
    # Open compressed file, load into dictionary
    with gzip.open(filename, "r") as f:
        data = json.load(f)
    
    # Load dictionary into DataFrame, set index
    df = pd.DataFrame().from_dict(data["observations"]).set_index("id")
   
    return df


In [ ]:
df = wrangle("data/taiwan-bankruptcy-data.json.gz")
print("df shape:", df.shape)
df.head()


In [ ]:
nans_by_col = df.isnull().sum() 
print("nans_by_col shape:", nans_by_col.shape)
nans_by_col.head()


In [ ]:
# Plot class balance
fig, ax = plt.subplots()
df['bankrupt'].value_counts(normalize=True).plot(
    kind="bar",
    xlabel="Bankrupt",
    ylabel="Frequency",
    title="Class Balance"
)


In [ ]:
target = "bankrupt"
X = df.drop(columns=target)
y = df[target]
print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size=0.2, random_state=42
)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


In [ ]:
over_sampler = RandomOverSampler(random_state=42)
X_train_over, y_train_over = over_sampler.fit_resample(X_train, y_train)
print("X_train_over shape:", X_train_over.shape)
X_train_over.head()


In [ ]:
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train_over, y_train_over)


In [ ]:
cv_scores = cross_val_score(
    clf, X_train_over,y_train_over, cv=5, n_jobs=-1
)
print(cv_scores)


In [ ]:
# Run this code cell

params = {
    "max_depth": range(30, 50, 10),
    "n_estimators": range(25, 51, 25),
}


In [ ]:
model = GridSearchCV(
    clf,
    param_grid=params,
    cv=5,
    n_jobs=-1,
    verbose=1
)
model


In [ ]:
model.fit(X_train_over, y_train_over)


In [ ]:
cv_results = pd.DataFrame(model.cv_results_)
cv_results.head(5)


In [ ]:
best_params = model.best_params_
print(best_params)


In [ ]:
acc_train = model.score(X_train, y_train)
acc_test = model.score(X_test, y_test)

print("Model Training Accuracy:", round(acc_train, 4))
print("Model Test Accuracy:", round(acc_test, 4))


In [ ]:
# Create figure and axis
fig, ax = plt.subplots()

# write code here
ConfusionMatrixDisplay.from_estimator(model,X_test, y_test, ax=ax )


In [ ]:
class_report = classification_report(y_test, model.predict(X_test))
print(class_report)


In [ ]:
fig, ax = plt.subplots()

# Get feature names from training data
features = X_train_over.columns

# Extract importances from model
importances = model.best_estimator_.feature_importances_

# Create a series with feature names and importances
feat_imp = pd.Series(importances, index=features).sort_values()
# Plot 10 most important features
feat_imp.tail(10).plot(kind="barh", ax=ax)

plt.xlabel("Gini Importance")
plt.ylabel("Feature")
plt.title("Feature Importance");


In [ ]:
# Save model
with open("model-5-5.pkl", "wb") as f:
    pickle.dump(model,f)


In [ ]:
# Import your module
from my_predictor_assignment import make_predictions

# Generate predictions
y_test_pred = make_predictions(
    data_filepath="data/taiwan-bankruptcy-data-test-features.json.gz",
    model_filepath="model-5-5.pkl",
)

print("predictions shape:", y_test_pred.shape)
y_test_pred.head()
